# Deteksi Penyakit Daun Kentang (Potato Disease Detection)
## Menggunakan CNN dengan Transfer Learning MobileNetV2

Notebook ini melakukan klasifikasi penyakit daun kentang menggunakan
Convolutional Neural Network (CNN) dengan Transfer Learning MobileNetV2.

**Kelas yang dideteksi:**
1. Early Blight (Penyakit Bercak Awal)
2. Late Blight (Penyakit Bercak Akhir)
3. Healthy (Sehat)

**Tahapan:**
1. Data Collection (Upload Manual)
2. Exploratory Data Analysis (EDA)
3. Data Processing
4. Feature Engineering
5. Modelling (CNN)
6. Evaluation
7. Simpan Model
8. Uji Coba Inference

In [ ]:
# CELL 1: Import Library
import os
import random
import zipfile
import shutil
import json
import warnings

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')

# Seed dibuat tetap agar pembagian data dan proses training lebih mudah direproduksi.
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Setup visualisasi.
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 8)

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")
print("Semua library berhasil diimport.")


---
## 1. Data Collection (Upload Manual)

Karena dataset dari Kaggle tidak terbaca otomatis, kita akan upload manual.

**Langkah-langkah:**
1. Download dataset dari: https://www.kaggle.com/datasets/arajmishra/potato-dataset
2. Ekstrak ZIP di laptop Anda
3. Upload folder `PlantVillage` ke Colab

**Atau** upload file ZIP langsung dan ekstrak di sini.

In [ ]:
# CELL 2: Upload Dataset (Manual)
from google.colab import files
import zipfile
import os

print("=" * 60)
print("UPLOAD DATASET")
print("=" * 60)
print("\nUpload file potato-dataset.zip atau folder PlantVillage yang sudah di-zip")
print("Download dari: https://www.kaggle.com/datasets/arajmishra/potato-dataset")
print("\nKlik tombol 'Choose Files' dan pilih file ZIP Anda\n")

uploaded = files.upload()

# Setiap file ZIP yang diunggah diekstrak ke direktori kerja Colab.
print("\nMengekstrak dataset...")
for filename in uploaded.keys():
    with zipfile.ZipFile(filename, 'r') as zip_ref:
        zip_ref.extractall('.')
    print(f"{filename} berhasil diekstrak.")

print("\nDataset siap digunakan.")


In [ ]:
# CELL 3: Setup dan Cek Dataset
print("=" * 60)
print("SETUP DAN CEK DATASET")
print("=" * 60)

# Cari folder PlantVillage secara otomatis.
IMG_DIR = None
for root, dirs, files in os.walk('.'):
    if 'PlantVillage' in root:
        IMG_DIR = root
        break

# Jika tidak ditemukan, cari folder pertama yang berisi file gambar.
if IMG_DIR is None:
    for root, dirs, files in os.walk('.'):
        if any(f.endswith(('.jpg', '.JPG', '.png', '.PNG')) for f in files):
            IMG_DIR = root
            break

# Alternatif terakhir: cari folder yang namanya mengandung kata Potato.
if IMG_DIR is None:
    for root, dirs, files in os.walk('.'):
        if 'Potato' in root or 'potato' in root:
            IMG_DIR = root
            break

print(f"Folder dataset ditemukan: {IMG_DIR}")

# Konfigurasi kelas harus sama untuk split, generator, evaluasi, dan inference.
CLASS_NAMES = ['Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy']
CLASS_LABELS = {
    'Potato___Early_blight': 'Early Blight (Bercak Awal)',
    'Potato___Late_blight': 'Late Blight (Bercak Akhir)',
    'Potato___healthy': 'Healthy (Sehat)'
}
NUM_CLASSES = len(CLASS_NAMES)

if IMG_DIR:
    print("\nIsi folder dataset:")
    print("-" * 50)

    items = os.listdir(IMG_DIR)
    subfolders = [f for f in items if os.path.isdir(os.path.join(IMG_DIR, f))]

    if subfolders:
        print(f"Subfolder ditemukan: {subfolders}\n")
        print("Distribusi dataset:")
        print("-" * 50)

        for folder in subfolders:
            folder_path = os.path.join(IMG_DIR, folder)
            images = [
                f for f in os.listdir(folder_path)
                if f.endswith(('.jpg', '.JPG', '.png', '.PNG'))
            ]
            count = len(images)

            # Gunakan pencocokan nama folder secara langsung terlebih dahulu.
            label = CLASS_LABELS.get(folder, "Unknown")

            # Fallback untuk dataset dengan penamaan folder yang sedikit berbeda.
            if label == "Unknown":
                for class_name in CLASS_NAMES:
                    clean_class = class_name.replace('Potato___', '').replace('_', ' ')
                    normalized_folder = folder.replace('_', ' ')
                    if (
                        clean_class.lower() in normalized_folder.lower()
                        or normalized_folder.lower() in clean_class.lower()
                    ):
                        label = CLASS_LABELS[class_name]
                        break

            print(f"{label:35s}: {count:5d} gambar")
    else:
        # Tidak ada subfolder; file gambar berada langsung di IMG_DIR.
        images = [
            f for f in items
            if f.endswith(('.jpg', '.JPG', '.png', '.PNG'))
        ]
        print(f"Tidak ada subfolder. Total gambar: {len(images)}")
        print("Dataset perlu disusun ulang dengan struktur folder per kelas.")
        print("\nMencoba mengatur ulang struktur dataset...")

        NEW_DIR = "potato_dataset_structured"
        for class_name in CLASS_NAMES:
            os.makedirs(os.path.join(NEW_DIR, class_name), exist_ok=True)

        for f in images:
            moved = False
            for class_name in CLASS_NAMES:
                clean_class = class_name.replace('Potato___', '').replace('_', ' ')
                if clean_class.lower() in f.lower():
                    shutil.move(
                        os.path.join(IMG_DIR, f),
                        os.path.join(NEW_DIR, class_name, f)
                    )
                    moved = True
                    break
            if not moved:
                os.makedirs(os.path.join(NEW_DIR, 'Unknown'), exist_ok=True)
                shutil.move(
                    os.path.join(IMG_DIR, f),
                    os.path.join(NEW_DIR, 'Unknown', f)
                )

        IMG_DIR = NEW_DIR
        print(f"Dataset disusun ulang di: {IMG_DIR}")

        print("\nDistribusi dataset setelah penyusunan ulang:")
        print("-" * 50)
        for class_name in CLASS_NAMES:
            folder_path = os.path.join(IMG_DIR, class_name)
            if os.path.exists(folder_path):
                count = len([
                    f for f in os.listdir(folder_path)
                    if f.endswith(('.jpg', '.JPG', '.png', '.PNG'))
                ])
                print(f"{CLASS_LABELS[class_name]:35s}: {count:5d} gambar")
else:
    print("Dataset tidak ditemukan.")
    print("\nPastikan Anda sudah upload file ZIP dan mengekstraknya.")
    print("Struktur folder yang diharapkan:")
    print("   PlantVillage/")
    print("   |-- Potato___Early_blight/")
    print("   |-- Potato___Late_blight/")
    print("   `-- Potato___healthy/")


In [ ]:
# CELL 4: Split Dataset (Versi Perbaikan)
print("=" * 60)
print("SPLIT DATASET (Train 70% | Val 10% | Test 20%)")
print("=" * 60)

from sklearn.model_selection import train_test_split
import shutil

DEST_DIR = "dataset"

# Hapus hasil split lama agar tidak ada file tersisa saat cell dijalankan ulang.
if os.path.exists(DEST_DIR):
    shutil.rmtree(DEST_DIR)

# Buat ulang struktur folder train, validation, dan test.
for split in ['train', 'val', 'test']:
    for class_name in CLASS_NAMES:
        os.makedirs(os.path.join(DEST_DIR, split, class_name), exist_ok=True)

if IMG_DIR and os.path.exists(IMG_DIR):
    print(f"\nSource: {IMG_DIR}")

    subfolders = [
        f for f in os.listdir(IMG_DIR)
        if os.path.isdir(os.path.join(IMG_DIR, f))
    ]

    if subfolders:
        print(f"Subfolder ditemukan: {subfolders}")

        # Mapping nama folder sumber ke nama kelas yang digunakan model.
        folder_to_class = {}
        for folder in subfolders:
            if folder in CLASS_NAMES:
                folder_to_class[folder] = folder
                continue

            normalized_folder = folder.replace('_', ' ').lower()
            for class_name in CLASS_NAMES:
                clean_class = class_name.replace('Potato___', '').replace('_', ' ').lower()
                if clean_class in normalized_folder or normalized_folder in clean_class:
                    folder_to_class[folder] = class_name
                    break

            if folder not in folder_to_class:
                for class_name in CLASS_NAMES:
                    if class_name.replace('Potato___', '') in folder:
                        folder_to_class[folder] = class_name
                        break

        print("\nMapping folder ke kelas:")
        for folder, class_name in folder_to_class.items():
            print(f"  {folder} -> {class_name}")

        print("\nMemisahkan data...")
        print("-" * 50)

        total_train = total_val = total_test = 0

        for folder, class_name in folder_to_class.items():
            if class_name not in CLASS_NAMES:
                print(f"{folder} tidak cocok dengan kelas yang diinginkan, dilewati")
                continue

            folder_path = os.path.join(IMG_DIR, folder)
            if not os.path.isdir(folder_path):
                continue

            images = [
                f for f in os.listdir(folder_path)
                if f.endswith(('.jpg', '.JPG', '.png', '.PNG'))
            ]

            if len(images) == 0:
                print(f"Tidak ada gambar di {folder}")
                continue

            # Pembagian per kelas menjaga proporsi 70% train, 10% val, dan 20% test.
            train_imgs, temp_imgs = train_test_split(
                images,
                test_size=0.30,
                random_state=SEED
            )
            val_imgs, test_imgs = train_test_split(
                temp_imgs,
                test_size=0.6667,
                random_state=SEED
            )

            # Salin gambar ke folder hasil split tanpa mengubah file sumber.
            for img in train_imgs:
                shutil.copy(
                    os.path.join(folder_path, img),
                    os.path.join(DEST_DIR, 'train', class_name, img)
                )
            for img in val_imgs:
                shutil.copy(
                    os.path.join(folder_path, img),
                    os.path.join(DEST_DIR, 'val', class_name, img)
                )
            for img in test_imgs:
                shutil.copy(
                    os.path.join(folder_path, img),
                    os.path.join(DEST_DIR, 'test', class_name, img)
                )

            label = CLASS_LABELS.get(class_name, class_name)
            print(
                f"{label:30s}: Train={len(train_imgs):4d}, "
                f"Val={len(val_imgs):4d}, Test={len(test_imgs):4d}"
            )

            total_train += len(train_imgs)
            total_val += len(val_imgs)
            total_test += len(test_imgs)

        print("-" * 50)
        print(
            f"{'TOTAL':30s}: Train={total_train:4d}, "
            f"Val={total_val:4d}, Test={total_test:4d}"
        )
        print("\nDataset split selesai.")
        print(f"Dataset tersimpan di folder: {DEST_DIR}")

        # Verifikasi jumlah file pada folder train untuk setiap kelas.
        print("\nVerifikasi hasil split:")
        for class_name in CLASS_NAMES:
            train_path = os.path.join(DEST_DIR, 'train', class_name)
            if os.path.exists(train_path):
                count = len([
                    f for f in os.listdir(train_path)
                    if f.endswith(('.jpg', '.JPG', '.png', '.PNG'))
                ])
                print(f"  {CLASS_LABELS[class_name]:30s}: {count:4d} gambar di train/")
    else:
        print("Tidak ditemukan subfolder di IMG_DIR")
else:
    print("IMG_DIR tidak ditemukan.")


---
## 2. Exploratory Data Analysis (EDA)

In [ ]:
# CELL 5: EDA - Distribusi Kelas
print("=" * 60)
print("EXPLORATORY DATA ANALYSIS (EDA)")
print("=" * 60)

# Hitung distribusi kelas pada data training.
class_dist = {}
for class_name in CLASS_NAMES:
    train_path = os.path.join(DEST_DIR, 'train', class_name)
    if os.path.exists(train_path):
        count = len([
            f for f in os.listdir(train_path)
            if f.endswith(('.jpg', '.JPG', '.png', '.PNG'))
        ])
        class_dist[class_name] = count
    else:
        class_dist[class_name] = 0

# Visualisasi distribusi data untuk melihat ketidakseimbangan kelas.
colors = ['#e74c3c', '#f39c12', '#2ecc71']
fig, ax = plt.subplots(figsize=(8, 5))
labels_display = [CLASS_LABELS.get(k, k) for k in class_dist.keys()]
bars = ax.bar(
    labels_display,
    class_dist.values(),
    color=colors,
    edgecolor='black',
    linewidth=1.5
)
ax.set_title('Distribusi Data per Kelas Penyakit Daun Kentang', fontsize=14, fontweight='bold')
ax.set_xlabel('Jenis Penyakit', fontsize=12)
ax.set_ylabel('Jumlah Gambar', fontsize=12)
ax.grid(True, alpha=0.3, axis='y')

for bar, val in zip(bars, class_dist.values()):
    if val > 0:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 3,
            str(val),
            ha='center',
            fontsize=11,
            fontweight='bold'
        )

plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print(f"\nTotal training data: {sum(class_dist.values())} gambar")
print("\nCatatan: Dataset tidak seimbang (imbalanced)")
print("   - Kelas 'Healthy' memiliki data paling sedikit")
print("   - Data Augmentation dan Class Weighting digunakan untuk membantu mengatasi imbalance")


In [ ]:
# CELL 6: EDA - Sample Gambar per Kelas (Versi Perbaikan)
print("\n" + "=" * 60)
print("SAMPLE GAMBAR PER KELAS")
print("=" * 60)

fig, axes = plt.subplots(1, len(CLASS_NAMES), figsize=(15, 5))
if len(CLASS_NAMES) == 1:
    axes = [axes]

for idx, class_name in enumerate(CLASS_NAMES):
    train_path = os.path.join(DEST_DIR, 'train', class_name)

    # Pastikan folder tersedia dan berisi setidaknya satu gambar.
    if os.path.exists(train_path):
        images = [
            f for f in os.listdir(train_path)
            if f.endswith(('.jpg', '.JPG', '.png', '.PNG'))
        ]

        if images:
            img_path = os.path.join(train_path, images[0])
            img = plt.imread(img_path)
            axes[idx].imshow(img)
            axes[idx].set_title(
                f"{CLASS_LABELS.get(class_name, class_name)}\n({len(images)} gambar)",
                fontsize=11,
                fontweight='bold'
            )
        else:
            axes[idx].text(
                0.5,
                0.5,
                f"Tidak ada gambar\ndi {class_name}",
                ha='center',
                va='center',
                fontsize=10,
                color='red'
            )
            axes[idx].set_title(
                CLASS_LABELS.get(class_name, class_name),
                fontsize=11,
                fontweight='bold'
            )
    else:
        axes[idx].text(
            0.5,
            0.5,
            f"Folder tidak ditemukan\n{class_name}",
            ha='center',
            va='center',
            fontsize=10,
            color='red'
        )
        axes[idx].set_title(
            CLASS_LABELS.get(class_name, class_name),
            fontsize=11,
            fontweight='bold'
        )

    axes[idx].axis('off')

plt.suptitle('Sample Citra Daun Kentang per Kelas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nStatus dataset:")
print("-" * 50)
for class_name in CLASS_NAMES:
    train_path = os.path.join(DEST_DIR, 'train', class_name)
    if os.path.exists(train_path):
        count = len([
            f for f in os.listdir(train_path)
            if f.endswith(('.jpg', '.JPG', '.png', '.PNG'))
        ])
        print(f"  {CLASS_LABELS.get(class_name, class_name):30s}: {count:4d} gambar")
    else:
        print(f"  {CLASS_LABELS.get(class_name, class_name):30s}: FOLDER TIDAK DITEMUKAN")


In [ ]:
# CELL 6A: EDA - Ukuran Citra dan Pemeriksaan Duplikasi
from collections import Counter, defaultdict
from PIL import Image
import hashlib

print("\n" + "=" * 60)
print("PEMERIKSAAN UKURAN CITRA DAN DUPLIKASI")
print("=" * 60)

image_records = []
image_sizes = []
corrupt_files = []
hash_groups = defaultdict(list)

# Pemeriksaan dilakukan pada seluruh hasil split untuk mendeteksi file rusak
# dan gambar identik yang mungkin muncul pada lebih dari satu subset data.
for split in ['train', 'val', 'test']:
    for class_name in CLASS_NAMES:
        class_dir = os.path.join(DEST_DIR, split, class_name)
        if not os.path.exists(class_dir):
            continue

        for filename in os.listdir(class_dir):
            if not filename.endswith(('.jpg', '.JPG', '.png', '.PNG')):
                continue

            file_path = os.path.join(class_dir, filename)
            try:
                with Image.open(file_path) as img:
                    image_sizes.append(img.size)
                    img.verify()

                with open(file_path, 'rb') as file_obj:
                    file_hash = hashlib.sha256(file_obj.read()).hexdigest()

                record = {
                    'split': split,
                    'class_name': class_name,
                    'filename': filename,
                    'path': file_path
                }
                image_records.append(record)
                hash_groups[file_hash].append(record)
            except Exception as error:
                corrupt_files.append((file_path, str(error)))

print(f"Total gambar yang diperiksa: {len(image_records)}")
print(f"File rusak atau tidak terbaca: {len(corrupt_files)}")

if image_sizes:
    size_counts = Counter(image_sizes)
    widths = [size[0] for size in image_sizes]
    heights = [size[1] for size in image_sizes]

    print(f"Ukuran minimum: {min(widths)} x {min(heights)} piksel")
    print(f"Ukuran maksimum: {max(widths)} x {max(heights)} piksel")
    print(f"Ukuran paling umum: {size_counts.most_common(1)[0][0]}")

    top_sizes = size_counts.most_common(10)
    size_labels = [f"{width}x{height}" for (width, height), _ in top_sizes]
    size_values = [count for _, count in top_sizes]

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(size_labels, size_values, edgecolor='black')
    ax.set_title('Sepuluh Ukuran Citra yang Paling Sering Muncul', fontweight='bold')
    ax.set_xlabel('Resolusi Citra')
    ax.set_ylabel('Jumlah Gambar')
    ax.tick_params(axis='x', rotation=45)
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()

# Kelompok hash dengan lebih dari satu file menunjukkan duplikat identik.
duplicate_groups = [records for records in hash_groups.values() if len(records) > 1]
cross_split_duplicates = [
    records for records in duplicate_groups
    if len({record['split'] for record in records}) > 1
]

print(f"Kelompok gambar duplikat identik: {len(duplicate_groups)}")
print(f"Duplikat yang melintasi train/val/test: {len(cross_split_duplicates)}")

if cross_split_duplicates:
    print("\nContoh duplikat lintas subset:")
    for group_number, records in enumerate(cross_split_duplicates[:5], start=1):
        locations = [
            f"{record['split']}/{record['class_name']}/{record['filename']}"
            for record in records
        ]
        print(f"  Grup {group_number}: {' | '.join(locations)}")
else:
    print("Tidak ditemukan duplikat identik yang melintasi subset data.")

if corrupt_files:
    print("\nContoh file yang bermasalah:")
    for file_path, error in corrupt_files[:5]:
        print(f"  {file_path}: {error}")


---
## 3. Data Processing

Semua gambar akan di-resize ke ukuran tetap, diproses menggunakan `preprocess_input` MobileNetV2,
dan dilakukan augmentasi data untuk mengatasi imbalanced dataset.

In [ ]:
# CELL 7: Data Processing & Augmentation
print("=" * 60)
print("DATA PREPROCESSING & AUGMENTATION")
print("=" * 60)

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 15

# MobileNetV2 pretrained menggunakan preprocess_input dengan rentang nilai [-1, 1].
# Augmentasi hanya diterapkan pada data training.
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)

# Validation dan test hanya menggunakan preprocessing yang sama tanpa augmentasi.
val_test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

print("Teknik Augmentasi yang digunakan:")
print("  1. Rotation (20 derajat)")
print("  2. Width/Height Shift (10%)")
print("  3. Zoom Range (10%)")
print("  4. Horizontal Flip")
print("  5. Brightness Adjustment")
print("  6. MobileNetV2 preprocess_input (rentang -1 sampai 1)")

# Generator membaca gambar berdasarkan folder kelas yang telah ditentukan.
train_generator = train_datagen.flow_from_directory(
    os.path.join(DEST_DIR, 'train'),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    classes=CLASS_NAMES,
    shuffle=True,
    seed=SEED
)

validation_generator = val_test_datagen.flow_from_directory(
    os.path.join(DEST_DIR, 'val'),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    classes=CLASS_NAMES,
    shuffle=False
)

test_generator = val_test_datagen.flow_from_directory(
    os.path.join(DEST_DIR, 'test'),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    classes=CLASS_NAMES,
    shuffle=False
)

print(f"\nTotal Training: {train_generator.samples} gambar")
print(f"Total Validation: {validation_generator.samples} gambar")
print(f"Total Test: {test_generator.samples} gambar")

# Class weight membantu memberi perhatian lebih pada kelas dengan data lebih sedikit.
from sklearn.utils import class_weight
train_labels = train_generator.classes
class_weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)
class_weight_dict = dict(enumerate(class_weights))

print("\nClass Weights:")
for name, weight in zip(CLASS_NAMES, class_weights):
    print(f"   {CLASS_LABELS[name]}: {weight:.3f}")


---
## 4. Feature Engineering

CNN akan mengekstrak fitur secara otomatis melalui:
- Convolutional layers: mendeteksi tepi, tekstur, pola penyakit
- Pooling layers: mereduksi dimensi
- Fully connected layers: klasifikasi

In [ ]:
# CELL 8: Feature Engineering
print("=" * 60)
print("FEATURE ENGINEERING")
print("=" * 60)

print("\nBagaimana CNN mempelajari fitur penyakit daun kentang:")
print("-" * 50)
print("Layer awal -> Deteksi edge, garis tepi daun, dan kontur dasar")
print("Layer menengah -> Deteksi bentuk bercak dan tekstur permukaan daun")
print("Layer lebih dalam -> Deteksi pola penyakit dan perubahan warna")
print("Global Average Pooling -> Merangkum feature map hasil MobileNetV2")
print("Fully Connected -> Klasifikasi berdasarkan kombinasi fitur")

# Ambil satu batch untuk memastikan dimensi input dan label sudah benar.
sample_batch, sample_labels = next(iter(train_generator))
print(f"\nShape batch: {sample_batch.shape}")
print(f"Shape label: {sample_labels.shape}")
print(f"Label range: {sample_labels.min()} - {sample_labels.max()}")


---
## 5. Modelling (CNN)

In [ ]:
# CELL 9: Build Model
print("=" * 60)
print("MEMBANGUN MODEL CNN")
print("=" * 60)

# MobileNetV2 digunakan sebagai feature extractor dengan bobot ImageNet.
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

# Bobot base model dibekukan pada tahap transfer learning pertama.
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(NUM_CLASSES, activation='softmax')
])

# Categorical crossentropy sesuai untuk klasifikasi multi-kelas one-hot encoded.
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("\nArsitektur Model:")
print("-" * 50)
model.summary()
print("-" * 50)
print(f"Total parameter: {model.count_params():,}")


In [ ]:
# CELL 10: Training Model
print("=" * 60)
print("MODEL TRAINING")
print("=" * 60)

print(f"\nTraining dimulai... ({EPOCHS} epoch)")
print("Menggunakan MobileNetV2 dengan Transfer Learning")
print("-" * 50)

# Semua callback menggunakan val_loss agar definisi model terbaik konsisten.
callbacks_list = [
    callbacks.EarlyStopping(
        monitor='val_loss',
        mode='min',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        mode='min',
        factor=0.5,
        patience=3,
        min_lr=0.00001,
        verbose=1
    ),
    callbacks.ModelCheckpoint(
        'best_model.keras',
        monitor='val_loss',
        mode='min',
        save_best_only=True,
        verbose=1
    )
]

print("Strategi Pencegahan Overfitting:")
print("  1. Train-Validation-Test Split (70%-10%-20%)")
print("  2. Data Augmentation")
print("  3. Dropout Layer (0.3)")
print("  4. Early Stopping")
print("  5. ReduceLROnPlateau")
print("  6. Class Weighting")

history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=validation_generator,
    callbacks=callbacks_list,
    class_weight=class_weight_dict,
    verbose=1
)

print("\nTRAINING SELESAI.")


In [ ]:
# CELL 11: Plot Training History
print("=" * 60)
print("GRAFIK TRAINING")
print("=" * 60)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Grafik accuracy membandingkan kemampuan belajar dan generalisasi model.
ax1.plot(history.history['accuracy'], 'b-', label='Training Accuracy', linewidth=2)
ax1.plot(history.history['val_accuracy'], 'r-', label='Validation Accuracy', linewidth=2)
ax1.set_title('Model Accuracy', fontsize=14, fontweight='bold')
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Accuracy', fontsize=12)
ax1.legend(loc='lower right')
ax1.grid(True, alpha=0.3)
ax1.set_ylim([0, 1])

# Grafik loss diperlukan karena overfitting tidak cukup dinilai dari accuracy saja.
ax2.plot(history.history['loss'], 'b-', label='Training Loss', linewidth=2)
ax2.plot(history.history['val_loss'], 'r-', label='Validation Loss', linewidth=2)
ax2.set_title('Model Loss', fontsize=14, fontweight='bold')
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Loss', fontsize=12)
ax2.legend(loc='upper right')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

final_acc = history.history['accuracy'][-1]
final_val_acc = history.history['val_accuracy'][-1]
final_loss = history.history['loss'][-1]
final_val_loss = history.history['val_loss'][-1]
accuracy_gap = final_acc - final_val_acc
best_val_loss = min(history.history['val_loss'])
best_val_epoch = int(np.argmin(history.history['val_loss']) + 1)
val_loss_increase = final_val_loss - best_val_loss

print("\nInterpretasi Grafik:")
print("-" * 50)
print(f"Final Training Accuracy: {final_acc:.4f}")
print(f"Final Validation Accuracy: {final_val_acc:.4f}")
print(f"Accuracy Gap: {accuracy_gap:.4f}")
print(f"Final Training Loss: {final_loss:.4f}")
print(f"Final Validation Loss: {final_val_loss:.4f}")
print(f"Best Validation Loss: {best_val_loss:.4f} pada epoch {best_val_epoch}")

# Aturan berikut digunakan sebagai indikator awal dan tetap perlu dibaca bersama grafik.
if final_acc < 0.70 and final_val_acc < 0.70:
    print("Kesimpulan awal: terdapat indikasi underfitting karena accuracy training dan validation masih rendah.")
elif accuracy_gap > 0.10:
    print("Kesimpulan awal: terdapat indikasi overfitting yang kuat karena gap accuracy besar.")
elif accuracy_gap > 0.05 or val_loss_increase > 0.05:
    print("Kesimpulan awal: terdapat indikasi overfitting ringan pada epoch akhir.")
else:
    print("Kesimpulan awal: tidak terdapat indikasi overfitting atau underfitting yang signifikan.")


---
## 6. Evaluation

In [ ]:
# CELL 12: Model Evaluation
print("=" * 60)
print("MODEL EVALUATION")
print("=" * 60)

# Muat checkpoint terbaik berdasarkan validation loss.
model = keras.models.load_model('best_model.keras')

# Test generator tidak diacak sehingga urutan prediksi sesuai dengan label aktual.
y_pred_probs = model.predict(test_generator)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = test_generator.classes

# Weighted metrics mengikuti proporsi kelas, sedangkan macro metrics memberi bobot sama.
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
macro_precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
macro_recall = recall_score(y_true, y_pred, average='macro', zero_division=0)
macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)

print("\nHasil Evaluasi Model:")
print("-" * 45)
print(f"Accuracy:           {accuracy:.4f} ({accuracy * 100:.2f}%)")
print(f"Weighted Precision: {precision:.4f}")
print(f"Weighted Recall:    {recall:.4f}")
print(f"Weighted F1-Score:  {f1:.4f}")
print(f"Macro Precision:    {macro_precision:.4f}")
print(f"Macro Recall:       {macro_recall:.4f}")
print(f"Macro F1-Score:     {macro_f1:.4f}")

print("\nClassification Report:")
print("-" * 60)
class_names_display = [CLASS_LABELS.get(name, name) for name in CLASS_NAMES]
print(
    classification_report(
        y_true,
        y_pred,
        target_names=class_names_display,
        zero_division=0
    )
)

# Confusion matrix memperlihatkan kesalahan prediksi untuk setiap kelas.
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=class_names_display,
    yticklabels=class_names_display,
    annot_kws={'size': 14}
)
ax.set_title('Confusion Matrix - Deteksi Penyakit Daun Kentang', fontsize=14, fontweight='bold')
ax.set_xlabel('Prediksi', fontsize=12)
ax.set_ylabel('Aktual', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
# CELL 13: Tampilkan Prediksi pada Sample Test
print("\nSample Prediksi pada Data Test:")

# Pilih sampel secara acak dan usahakan setiap kelas terwakili.
rng = np.random.default_rng(SEED)
all_indices = np.arange(test_generator.samples)
selected_indices = []

for class_index in range(NUM_CLASSES):
    class_indices = np.where(test_generator.classes == class_index)[0]
    if len(class_indices) > 0:
        selected_indices.append(int(rng.choice(class_indices)))

remaining_count = min(8, test_generator.samples) - len(selected_indices)
remaining_pool = np.setdiff1d(all_indices, np.array(selected_indices, dtype=int))
if remaining_count > 0 and len(remaining_pool) > 0:
    selected_indices.extend(
        rng.choice(
            remaining_pool,
            size=min(remaining_count, len(remaining_pool)),
            replace=False
        ).tolist()
    )

fig, axes = plt.subplots(2, 4, figsize=(16, 9))
axes = axes.flatten()

for plot_index, sample_index in enumerate(selected_indices):
    image_path = test_generator.filepaths[sample_index]
    true_label = int(test_generator.classes[sample_index])

    # Tampilkan gambar asli, tetapi gunakan preprocess_input saat melakukan prediksi.
    display_img = keras.utils.load_img(image_path, target_size=(IMG_SIZE, IMG_SIZE))
    img_array = keras.utils.img_to_array(display_img)
    model_input = preprocess_input(np.expand_dims(img_array.copy(), axis=0))

    pred_probs = model.predict(model_input, verbose=0)
    pred_label = int(np.argmax(pred_probs[0]))
    confidence = float(pred_probs[0][pred_label])

    axes[plot_index].imshow(display_img)
    title_color = 'green' if pred_label == true_label else 'red'
    axes[plot_index].set_title(
        f'Aktual: {CLASS_LABELS[CLASS_NAMES[true_label]].split("(")[0].strip()}\n'
        f'Pred: {CLASS_LABELS[CLASS_NAMES[pred_label]].split("(")[0].strip()}\n'
        f'Conf: {confidence * 100:.1f}%',
        color=title_color,
        fontsize=10,
        fontweight='bold'
    )
    axes[plot_index].axis('off')

# Sembunyikan subplot yang tidak terpakai jika jumlah test kurang dari delapan.
for plot_index in range(len(selected_indices), len(axes)):
    axes[plot_index].axis('off')

plt.suptitle('Prediksi pada Test Set', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


---
## 7. Simpan Model

In [ ]:
# CELL 14: Simpan Model
print("=" * 60)
print("SIMPAN MODEL")
print("=" * 60)

# Folder ini menampung model dan konfigurasi kelas untuk proses deployment.
os.makedirs('Model', exist_ok=True)

# Simpan model terbaik dalam format H5 sesuai ketentuan tugas.
model_path = 'Model/potato_disease_model.h5'
model.save(model_path)

# Format Keras juga disimpan sebagai cadangan format native TensorFlow/Keras.
model.save('Model/potato_disease_model.keras')

config = {
    'classes': CLASS_NAMES,
    'class_labels': CLASS_LABELS,
    'num_classes': NUM_CLASSES,
    'img_size': IMG_SIZE,
    'test_accuracy': round(accuracy, 4)
}

with open('Model/model_config.json', 'w') as f:
    json.dump(config, f, indent=2)

size_mb = os.path.getsize(model_path) / (1024 * 1024)
print(f"Model tersimpan: {model_path}")
print(f"Ukuran: {size_mb:.2f} MB")
print("Class info tersimpan: Model/model_config.json")

print("\nFile di folder Model/:")
for filename in sorted(os.listdir('Model')):
    file_path = os.path.join('Model', filename)
    size = os.path.getsize(file_path) / 1024
    print(f"  {filename:30s} {size:10.2f} KB")


In [ ]:
# CELL 15: Download Model ke Laptop
from google.colab import files

print("=" * 60)
print("DOWNLOAD MODEL")
print("=" * 60)

print("\nMendownload model ke komputer Anda...")
files.download('Model/potato_disease_model.h5')
files.download('Model/model_config.json')

print("\nModel berhasil didownload.")
print("Model siap digunakan di aplikasi Flask.")


---
## 8. Uji Coba Inference

In [ ]:
# CELL 16: Load Model dan Konfigurasi
print("=" * 60)
print("LOAD MODEL")
print("=" * 60)

# Model dan konfigurasi dimuat kembali untuk menguji hasil export.
loaded_model = keras.models.load_model('Model/potato_disease_model.h5')

with open('Model/model_config.json', 'r') as f:
    cfg = json.load(f)

print("Model loaded successfully.")
print(f"Classes: {cfg['classes']}")
print(f"Test accuracy: {cfg['test_accuracy'] * 100:.2f}%")


In [ ]:
# CELL 17: Fungsi Prediksi
from tensorflow.keras.preprocessing.image import load_img, img_to_array

def predict_potato_disease(image_path, model=loaded_model, cfg=cfg):
    # Mengembalikan label, confidence, dan probabilitas untuk setiap kelas.
    # Ukuran input diambil dari file konfigurasi agar fungsi tidak bergantung
    # pada variabel global yang mungkin belum dijalankan.
    img_size = int(cfg['img_size'])
    img = load_img(image_path, target_size=(img_size, img_size))
    img_array = img_to_array(img)

    # Preprocessing inference harus sama dengan preprocessing saat training.
    img_array = preprocess_input(img_array)
    img_array = np.expand_dims(img_array, axis=0)

    predictions = model.predict(img_array, verbose=0)
    predicted_idx = int(np.argmax(predictions[0]))
    confidence = float(predictions[0][predicted_idx])

    predicted_class = cfg['classes'][predicted_idx]
    predicted_label = cfg['class_labels'][predicted_class]

    prob_per_class = {}
    for index, class_name in enumerate(cfg['classes']):
        label = cfg['class_labels'][class_name]
        prob_per_class[label] = float(predictions[0][index])

    return {
        'disease': predicted_label,
        'disease_key': predicted_class,
        'confidence': confidence,
        'confidence_pct': f"{confidence * 100:.2f}%",
        'probabilities': prob_per_class
    }

print("Fungsi prediksi siap.")


In [ ]:
# CELL 18: Upload Gambar dan Prediksi
from google.colab import files
from IPython.display import display, Image

print("=" * 60)
print("UPLOAD GAMBAR DAN PREDIKSI")
print("=" * 60)
print("\nUpload gambar daun kentang untuk diuji")
print("Klik 'Choose Files' dan pilih gambar\n")

uploaded = files.upload()

for filename in uploaded.keys():
    print(f"\nGambar: {filename}")
    display(Image(filename, width=200))

    result = predict_potato_disease(filename)

    print("\n" + "=" * 50)
    print("HASIL PREDIKSI")
    print("=" * 50)
    print(f"Penyakit: {result['disease']}")
    print(f"Confidence: {result['confidence_pct']}")

    print("\nProbabilitas per kelas:")
    print("-" * 40)
    for label, prob in result['probabilities'].items():
        bar = '█' * int(prob * 40)
        print(f"   {label:35s}: {prob * 100:5.1f}% {bar}")


---
## Kesimpulan

CNN dengan Transfer Learning MobileNetV2 berhasil digunakan untuk
klasifikasi tiga kondisi daun kentang.

**Hasil evaluasi terakhir:**

- Test Accuracy: **98,85%**
- Weighted Precision: **98,86%**
- Weighted Recall: **98,85%**
- Weighted F1-score: **98,85%**
- Macro F1-score: **99,17%**
- Prediksi benar: **428 dari 433 gambar test**

Model tidak menunjukkan underfitting maupun overfitting yang signifikan.
Validation loss sempat berfluktuasi, tetapi hasil test tetap tinggi.

Model disimpan dalam format `.h5`, `.keras`, dan konfigurasi `.json`.